In [15]:
import polars as pl
import os

from pathlib import Path

In [16]:
WORKING_PATH = Path('/group/pmc021/amunif/epi-thesis/workflow/17_Pairwise Ranking HepG2 GSE76344/')
DATASET_PATH = WORKING_PATH / 'dataset'
OUTPUT_PATH  = WORKING_PATH / 'output' / 'combined'

# Merge the experiment results

In [17]:
# Read all CSV into single dataframe
pl_df = pl.read_csv(OUTPUT_PATH / 'test' / "*-test-metrics.csv")

In [18]:
pl_df

model,seed,histone_marker,epochs_trained,val_accuracy,val_auc,test_accuracy,test_auc,test_aucprc,test_precision,test_recall,test_f1,antisymmetry
str,i64,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""DirectRanker""",1011,"""H3K4me3""",42,67.5,0.77,67.2,0.7593,0.7273,0.6434,0.7231,0.6809,0.903
"""LogisticRegression""",1011,"""H3K4me3""",null,69.5,0.7562,70.5,0.7447,0.6833,0.7234,0.6322,0.6748,0.752
"""RandomForest""",1011,"""H3K4me3""",null,70.3,0.7907,69.5,0.775,0.7539,0.7039,0.6384,0.6696,0.741
"""SVM_Linear""",1011,"""H3K4me3""",null,69.3,0.756,70.6,0.745,0.6838,0.7251,0.6322,0.6755,0.751
"""DirectRanker""",123,"""H3K4me3""",46,66.3,0.74,68.3,0.7495,0.7037,0.6508,0.7386,0.6919,0.902
…,…,…,…,…,…,…,…,…,…,…,…,…
"""SVM_Linear""",456,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",null,66.7,0.7301,64.7,0.6934,0.6352,0.6401,0.5903,0.6142,0.702
"""DirectRanker""",789,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",100,68.1,0.75,70.8,0.7875,0.76,0.681,0.7296,0.7045,0.976
"""LogisticRegression""",789,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",null,65.5,0.7029,68.3,0.7319,0.6839,0.6802,0.6331,0.6558,0.701


In [19]:
# Save the results by model 
MODEL_RESULT_FOLDER = OUTPUT_PATH / 'model_results'
os.makedirs(MODEL_RESULT_FOLDER, exist_ok=True)

for model_name in pl_df["model"].unique():
    pl_df.filter(pl_df["model"] == model_name).write_csv(MODEL_RESULT_FOLDER / f"{model_name}.csv", include_header=True)

In [20]:
# Save the results for all models and all seeds
pl_df.write_csv(OUTPUT_PATH / "GSE76344_all_results.csv", include_header=True)

In [31]:
summary_df = (
    pl_df
    .group_by(["histone_marker", "model"])
    .agg([
        pl.col("val_accuracy").mean().alias("val_accuracy_mean"),
        pl.col("val_accuracy").std().alias("val_accuracy_std"),
        pl.col("val_auc").mean().alias("val_auc_mean"),
        pl.col("val_auc").std().alias("val_auc_std"),
        pl.col("test_accuracy").mean().alias("test_accuracy_mean"),
        pl.col("test_accuracy").std().alias("test_accuracy_std"),
        pl.col("test_auc").mean().alias("test_auc_mean"),
        pl.col("test_auc").std().alias("test_auc_std"),
        pl.col("test_aucprc").mean().alias("test_aucprc_mean"),
        pl.col("test_aucprc").std().alias("test_aucprc_std"),
        pl.col("antisymmetry").mean().alias("antisymmetry_mean"),
        pl.col("antisymmetry").std().alias("antisymmetry_std")
    ])
    .sort(["histone_marker", "test_accuracy_mean"], descending=[False, True])
    .with_columns(pl.col(pl.Float64).round(4))
)

In [32]:
summary_df

histone_marker,model,val_accuracy_mean,val_accuracy_std,val_auc_mean,val_auc_std,test_accuracy_mean,test_accuracy_std,test_auc_mean,test_auc_std,test_aucprc_mean,test_aucprc_std,antisymmetry_mean,antisymmetry_std
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""H3K27ac""","""RandomForest""",70.16,1.6134,0.7778,0.0214,70.9,1.9545,0.7828,0.0197,0.7434,0.0226,0.7262,0.0101
"""H3K27ac""","""LogisticRegression""",68.4,1.8398,0.7303,0.024,68.22,1.63,0.7266,0.0215,0.6744,0.0205,0.7192,0.0109
"""H3K27ac""","""SVM_Linear""",68.38,1.8472,0.7309,0.0239,68.1,2.1783,0.7276,0.0208,0.6743,0.0199,0.7112,0.0192
"""H3K27ac""","""DirectRanker""",67.76,2.2546,0.758,0.0239,67.6,1.2629,0.7612,0.0215,0.7219,0.0192,0.8722,0.0043
"""H3K27ac-H3K27me3""","""RandomForest""",70.32,2.4641,0.7803,0.0219,70.46,2.3115,0.7861,0.0224,0.7534,0.0201,0.7504,0.0076
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""H3K9me3-H3K27ac-H3K27me3""","""SVM_Linear""",66.12,1.8647,0.711,0.023,66.3,2.1691,0.7075,0.0206,0.6647,0.0252,0.7178,0.0206
"""H3K9me3-H3K27me3""","""RandomForest""",65.04,2.305,0.6996,0.0198,65.92,2.2554,0.7088,0.0308,0.6607,0.0257,0.7394,0.0209
"""H3K9me3-H3K27me3""","""DirectRanker""",64.04,1.4639,0.7,0.02,64.26,2.3586,0.7088,0.0334,0.6707,0.0311,0.8876,0.0141


In [23]:
summary_df.write_csv(OUTPUT_PATH/ f"HepG2 GSE76344 Ranking.csv", include_header=True)

In [24]:
for model_name in summary_df["model"].unique():
    summary_df.filter(summary_df["model"] == model_name).write_csv(MODEL_RESULT_FOLDER / f"{model_name}_summary.csv", include_header=True)

# Merge the label distribution

In [25]:
# Read all label distribution CSV into single dataframe
label_df = pl.read_csv(OUTPUT_PATH / 'test' / "*-label-distribution.csv")

In [26]:
label_df

seed,histone_marker,split,label,count,total,percentage
i64,str,str,i64,i64,i64,f64
1011,"""H3K4me3""","""Train""",0,4208,8000,52.6
1011,"""H3K4me3""","""Train""",1,3792,8000,47.4
1011,"""H3K4me3""","""Val""",0,516,1000,51.6
1011,"""H3K4me3""","""Val""",1,484,1000,48.4
1011,"""H3K4me3""","""Test""",0,516,1000,51.6
…,…,…,…,…,…,…
789,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…","""Train""",1,3743,8000,46.79
789,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…","""Val""",0,535,1000,53.5
789,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…","""Val""",1,465,1000,46.5


In [27]:
# Make the summary by seed, split and label
label_summary_df = (
    label_df
    .group_by(['split', 'label'])
    .agg(pl.col('percentage').mean().round(2).alias('average_percentage_%'))
    .sort(['split', 'label'])
)

In [28]:
label_summary_df

split,label,average_percentage_%
str,i64,f64
"""Test""",0,52.32
"""Test""",1,47.68
"""Train""",0,52.72
"""Train""",1,47.28
"""Val""",0,51.44
"""Val""",1,48.56
